In [2]:
import numpy as np

class RunningMeanStd:
    """
    Tracks running mean and standard deviation using Welford's online algorithm.
    This is numerically stable and commonly used in RL frameworks like OpenAI Baselines.
    """
    def __init__(self, epsilon=1e-4, shape=()):
        """
        Args:
            epsilon: Small constant for numerical stability when normalizing
            shape: Shape of the data (e.g., () for scalar, (4,) for 4D vector)
        """
        self.mean = np.zeros(shape, dtype=np.float64)
        self.var = np.ones(shape, dtype=np.float64)
        self.count = epsilon  # Start with small count to avoid division by zero
    
    def update(self, x):
        """
        Update statistics with new batch of data.
        
        Args:
            x: New data, shape can be (shape,) or (batch_size, *shape)
        """
        batch_mean = np.mean(x, axis=0)
        batch_var = np.var(x, axis=0)
        batch_count = x.shape[0] if x.ndim > len(self.mean.shape) else 1
        
        self.update_from_moments(batch_mean, batch_var, batch_count)
    
    def update_from_moments(self, batch_mean, batch_var, batch_count):
        """
        Update from precomputed batch statistics (useful for distributed training).
        Uses parallel algorithm for combining statistics.
        """
        delta = batch_mean - self.mean
        total_count = self.count + batch_count
        
        new_mean = self.mean + delta * batch_count / total_count
        m_a = self.var * self.count
        m_b = batch_var * batch_count
        M2 = m_a + m_b + np.square(delta) * self.count * batch_count / total_count
        new_var = M2 / total_count
        
        self.mean = new_mean
        self.var = new_var
        self.count = total_count
    
    def normalize(self, x, clip_range=None):
        """
        Normalize data using current statistics.
        
        Args:
            x: Data to normalize
            clip_range: If provided, clip normalized values to [-clip_range, clip_range]
        
        Returns:
            Normalized data
        """
        normalized = (x - self.mean) / np.sqrt(self.var + 1e-8)
        if clip_range is not None:
            normalized = np.clip(normalized, -clip_range, clip_range)
        return normalized


In [3]:
# Demo usage
if __name__ == "__main__":
    print("=== Example 1: Scalar values ===")
    rms = RunningMeanStd()
    
    # Simulate streaming data from a changing distribution
    for i in range(5):
        # Generate batch of data with increasing mean
        data = np.random.randn(100) + i * 0.5
        rms.update(data)
        print(f"Step {i+1}: mean={rms.mean:.3f}, std={np.sqrt(rms.var):.3f}")
    
    # Normalize new data
    test_data = np.array([0.5, 1.5, 2.5])
    normalized = rms.normalize(test_data, clip_range=10.0)
    print(f"\nNormalized values: {normalized}")
    
    print("\n=== Example 2: Vector observations (e.g., RL states) ===")
    rms_vec = RunningMeanStd(shape=(4,))
    
    # Simulate batches of 4D state vectors
    for i in range(3):
        states = np.random.randn(50, 4) * (i + 1)  # Changing scale
        rms_vec.update(states)
        print(f"Step {i+1}: mean={rms_vec.mean}, std={np.sqrt(rms_vec.var)}")
    
    # Normalize a single state
    test_state = np.array([1.0, -1.0, 0.5, -0.5])
    normalized_state = rms_vec.normalize(test_state)
    print(f"\nNormalized state: {normalized_state}")

=== Example 1: Scalar values ===
Step 1: mean=0.180, std=1.016
Step 2: mean=0.341, std=1.057
Step 3: mean=0.559, std=1.080
Step 4: mean=0.792, std=1.128
Step 5: mean=1.032, std=1.201

Normalized values: [-0.44338067  0.38938826  1.22215718]

=== Example 2: Vector observations (e.g., RL states) ===
Step 1: mean=[ 0.11981098 -0.09700917  0.03217106  0.04770446], std=[1.09487502 1.16598109 0.95294644 1.25361955]
Step 2: mean=[-0.01411215 -0.36197107 -0.20344287  0.10773407], std=[1.64635887 1.43620227 1.72528207 1.72105145]
Step 3: mean=[-0.09098287 -0.10988715 -0.21937918  0.24857314], std=[1.85190462 2.20238698 2.16843364 2.15645281]

Normalized state: [ 0.58911396 -0.40415824  0.33175061 -0.34713171]


In [ ]:
# # Initialize
# env = gym.make('CartPole-v1')
# policy = PolicyNetwork()
# obs_rms = RunningMeanStd(shape=env.observation_space.shape)

# for episode in range(num_episodes):
#     obs = env.reset()
    
#     # 1. NORMALIZE using current statistics (even if poor initially)
#     normalized_obs = obs_rms.normalize(obs, clip_range=10)
    
#     # 2. Use normalized observation
#     action = policy.get_action(normalized_obs)
    
#     # 3. Collect trajectory
#     observations = []
#     for step in range(max_steps):
#         observations.append(obs)
#         obs, reward, done, _ = env.step(action)
        
#         # Normalize for next action
#         normalized_obs = obs_rms.normalize(obs, clip_range=10)
#         action = policy.get_action(normalized_obs)
        
#         if done:
#             break
    
#     # 4. UPDATE statistics with collected data
#     obs_rms.update(np.array(observations))
    
#     # 5. Train policy using normalized observations
#     # (normalize the batch again with updated statistics if needed)

In [ ]:
# import numpy as np
# from typing import Optional, Tuple, Union


# class RunningNormalizer:
#     """
#     Robust running normalization for non-stationary environments.
#     Combines Welford's online algorithm with exponential moving average (EMA)
#     for adaptive normalization in changing distributions.
#     """
    
#     def __init__(
#         self,
#         shape: Union[int, Tuple[int, ...]],
#         epsilon: float = 1e-8,
#         clip_range: Optional[Tuple[float, float]] = (-5.0, 5.0),
#         momentum: float = 0.99,
#         use_ema: bool = True,
#         min_samples: int = 2
#     ):
#         """
#         Initialize the running normalizer.
        
#         Args:
#             shape: Shape of the input data (int or tuple of ints)
#             epsilon: Small constant for numerical stability
#             clip_range: Optional range for clipping normalized values
#             momentum: Momentum factor for EMA (higher = slower adaptation)
#             use_ema: Whether to use EMA or standard running average
#             min_samples: Minimum samples before normalization is applied
#         """
#         if isinstance(shape, int):
#             shape = (shape,)
        
#         self.shape = shape
#         self.epsilon = epsilon
#         self.clip_range = clip_range
#         self.momentum = momentum
#         self.use_ema = use_ema
#         self.min_samples = min_samples
        
#         # Initialize statistics
#         self.mean = np.zeros(shape, dtype=np.float64)
#         self.var = np.ones(shape, dtype=np.float64)
#         self.std = np.ones(shape, dtype=np.float64)
#         self.count = 0
        
#         # For Welford's algorithm (numerical stability)
#         self.M2 = np.zeros(shape, dtype=np.float64)
        
#     def update(self, x: np.ndarray, batch_update: bool = False) -> None:
#         """
#         Update running statistics with new observation(s).
        
#         Args:
#             x: Input data (single sample or batch)
#             batch_update: If True, x is treated as a batch
#         """
#         x = np.asarray(x, dtype=np.float64)
        
#         if batch_update and x.ndim > len(self.shape):
#             # Batch update
#             batch_size = x.shape[0]
#             batch_mean = np.mean(x, axis=0)
#             batch_var = np.var(x, axis=0)
            
#             if self.use_ema and self.count > 0:
#                 # Exponential moving average update
#                 self.mean = self.momentum * self.mean + (1 - self.momentum) * batch_mean
#                 self.var = self.momentum * self.var + (1 - self.momentum) * batch_var
#             else:
#                 # Standard running average update
#                 delta = batch_mean - self.mean
#                 total_count = self.count + batch_size
                
#                 self.mean += delta * batch_size / total_count
                
#                 # Update variance (parallel algorithm)
#                 self.M2 += batch_var * batch_size + delta**2 * self.count * batch_size / total_count
#                 self.var = self.M2 / total_count if total_count > 1 else np.ones_like(self.var)
            
#             self.count += batch_size
            
#         else:
#             # Single sample update (Welford's algorithm)
#             if x.shape != self.shape:
#                 x = x.reshape(self.shape)
            
#             self.count += 1
            
#             if self.use_ema and self.count > 1:
#                 # EMA update for single sample
#                 delta = x - self.mean
#                 self.mean += (1 - self.momentum) * delta
#                 self.var = self.momentum * self.var + (1 - self.momentum) * delta**2
#             else:
#                 # Welford's update for numerical stability
#                 delta = x - self.mean
#                 self.mean += delta / self.count
#                 delta2 = x - self.mean
#                 self.M2 += delta * delta2
                
#                 if self.count > 1:
#                     self.var = self.M2 / (self.count - 1)
        
#         # Update standard deviation
#         self.std = np.sqrt(self.var + self.epsilon)
    
#     def normalize(self, x: np.ndarray) -> np.ndarray:
#         """
#         Normalize input using running statistics.
        
#         Args:
#             x: Input data to normalize
            
#         Returns:
#             Normalized data
#         """
#         if self.count < self.min_samples:
#             # Not enough samples yet, return unchanged or scaled
#             return x
        
#         x = np.asarray(x, dtype=np.float32)
#         normalized = (x - self.mean) / self.std
        
#         if self.clip_range is not None:
#             normalized = np.clip(normalized, self.clip_range[0], self.clip_range[1])
        
#         return normalized
    
#     def denormalize(self, x: np.ndarray) -> np.ndarray:
#         """
#         Denormalize data back to original scale.
        
#         Args:
#             x: Normalized data
            
#         Returns:
#             Denormalized data
#         """
#         return x * self.std + self.mean
    
#     def reset(self) -> None:
#         """Reset all statistics."""
#         self.mean = np.zeros(self.shape, dtype=np.float64)
#         self.var = np.ones(self.shape, dtype=np.float64)
#         self.std = np.ones(self.shape, dtype=np.float64)
#         self.count = 0
#         self.M2 = np.zeros(self.shape, dtype=np.float64)
    
#     def get_stats(self) -> dict:
#         """Get current statistics."""
#         return {
#             'mean': self.mean.copy(),
#             'var': self.var.copy(),
#             'std': self.std.copy(),
#             'count': self.count
#         }
    
#     def set_stats(self, stats: dict) -> None:
#         """Set statistics from a dictionary."""
#         self.mean = stats['mean'].copy()
#         self.var = stats['var'].copy()
#         self.std = stats['std'].copy()
#         self.count = stats['count']


# class VectorizedRunningNormalizer:
#     """
#     Vectorized version for multiple parallel environments (common in RL).
#     Maintains separate statistics for each environment.
#     """
    
#     def __init__(
#         self,
#         num_envs: int,
#         obs_shape: Union[int, Tuple[int, ...]],
#         **kwargs
#     ):
#         """
#         Initialize vectorized normalizer.
        
#         Args:
#             num_envs: Number of parallel environments
#             obs_shape: Shape of single observation
#             **kwargs: Additional arguments passed to RunningNormalizer
#         """
#         self.num_envs = num_envs
#         self.normalizers = [
#             RunningNormalizer(obs_shape, **kwargs) 
#             for _ in range(num_envs)
#         ]
    
#     def update(self, obs: np.ndarray, env_indices: Optional[np.ndarray] = None) -> None:
#         """
#         Update statistics for specified environments.
        
#         Args:
#             obs: Observations from environments (shape: [num_envs, ...])
#             env_indices: Which environments to update (None = all)
#         """
#         if env_indices is None:
#             env_indices = range(self.num_envs)
        
#         for i, idx in enumerate(env_indices):
#             self.normalizers[idx].update(obs[i])
    
#     def normalize(self, obs: np.ndarray) -> np.ndarray:
#         """Normalize observations from all environments."""
#         return np.array([
#             self.normalizers[i].normalize(obs[i])
#             for i in range(min(len(obs), self.num_envs))
#         ])
    
#     def synchronize(self) -> None:
#         """Synchronize statistics across all environments (useful for shared normalization)."""
#         if self.num_envs == 0:
#             return
        
#         # Compute weighted average of statistics
#         total_count = sum(n.count for n in self.normalizers)
#         if total_count == 0:
#             return
        
#         avg_mean = sum(n.mean * n.count for n in self.normalizers) / total_count
#         avg_var = sum(n.var * n.count for n in self.normalizers) / total_count
        
#         # Update all normalizers with averaged statistics
#         for normalizer in self.normalizers:
#             normalizer.mean = avg_mean.copy()
#             normalizer.var = avg_var.copy()
#             normalizer.std = np.sqrt(avg_var + normalizer.epsilon)
#             normalizer.count = total_count


# # Example usage for RL
# def example_rl_usage():
#     """Example of using running normalization in an RL setting."""
    
#     # Single environment
#     normalizer = RunningNormalizer(
#         shape=(84, 84, 3),  # e.g., image observation
#         epsilon=1e-8,
#         clip_range=(-5, 5),
#         momentum=0.99,  # Slower adaptation for stability
#         use_ema=True
#     )
    
#     # Simulate environment loop
#     for episode in range(100):
#         obs = np.random.randn(84, 84, 3)  # Simulated observation
        
#         # Update statistics
#         normalizer.update(obs)
        
#         # Get normalized observation for policy
#         norm_obs = normalizer.normalize(obs)
        
#         print(f"Episode {episode}: mean={normalizer.mean.mean():.4f}, "
#               f"std={normalizer.std.mean():.4f}")
    
#     # Vectorized environments
#     vec_normalizer = VectorizedRunningNormalizer(
#         num_envs=4,
#         obs_shape=(10,),  # e.g., state vector
#         momentum=0.995,
#         use_ema=True
#     )
    
#     # Simulate parallel environment steps
#     for step in range(1000):
#         obs_batch = np.random.randn(4, 10)  # 4 environments
        
#         # Update and normalize
#         vec_normalizer.update(obs_batch)
#         norm_obs_batch = vec_normalizer.normalize(obs_batch)
        
#         # Periodically synchronize statistics across environments
#         if step % 100 == 0:
#             vec_normalizer.synchronize()


# if __name__ == "__main__":
#     # Test the implementation
#     normalizer = RunningNormalizer(shape=(3,))
    
#     # Generate data with changing mean and variance
#     for i in range(1000):
#         # Simulate non-stationary data
#         mean_shift = i / 100
#         variance = 1 + i / 500
#         data = np.random.randn(3) * np.sqrt(variance) + mean_shift
        
#         normalizer.update(data)
        
#         if i % 100 == 0:
#             stats = normalizer.get_stats()
#             print(f"Step {i}: mean={stats['mean'].mean():.3f}, "
#                   f"std={stats['std'].mean():.3f}")
    
#     print("\nTesting batch update:")
#     batch_normalizer = RunningNormalizer(shape=(5,))
#     batch_data = np.random.randn(100, 5)
#     batch_normalizer.update(batch_data, batch_update=True)
#     print(f"Batch stats: mean={batch_normalizer.mean.mean():.3f}, "
#           f"std={batch_normalizer.std.mean():.3f}")

Step 0: mean=0.369, std=1.000
Step 100: mean=0.479, std=1.232
Step 200: mean=1.212, std=1.405
Step 300: mean=2.000, std=1.405
Step 400: mean=2.987, std=1.561
Step 500: mean=4.009, std=1.717
Step 600: mean=5.078, std=1.774
Step 700: mean=6.109, std=1.854
Step 800: mean=7.095, std=1.806
Step 900: mean=7.922, std=1.895

Testing batch update:
Batch stats: mean=-0.040, std=1.019


# pack env test

In [4]:
from src.normalizer import RunningMeanStd
from src.model import FiLMResNet
from src.envpacker import packenv
import torch

hidden_dim = 64
num_res_blocks = 2
output_dim = 3 # a/m, h, \mu

AGENTS = 10

model = FiLMResNet(
    AGENTS=AGENTS,
    hidden_dim=hidden_dim,
    num_res_blocks=num_res_blocks,
    output_dim=output_dim,
    dropout=0.1
)


# Create dummy data with explicit float32 dtype
money_disposable = (torch.randn(AGENTS, 1) * 100 + 500).float()  # Convert to float32
v = (torch.randn(AGENTS, 1) * 10 + 50).float()  # Convert to float32
tax_params = torch.tensor([0.1, 0.2, 0.3, 0.15, 0.25], dtype=torch.float32)  # Explicit dtype
rms = RunningMeanStd(shape=(2,))

packed_tensor, updated_rms = packenv(money_disposable, v, tax_params, rms)

# Ensure packed_tensor is float32
packed_tensor = packed_tensor.float()

model_output = model(packed_tensor)
print("Model output shape:", model_output.shape)  # Should be (AGENTS, output_dim)


Model output shape: torch.Size([10, 3])
